# Movie Recommendation — Full Pipeline

**Run all cells in order**, then start the app: `streamlit run app.py`

| Step | What it does |
|------|----------------|
| 1 | Setup & constants |
| 2 | Load language CSVs |
| 3 | TMDB online fill (auto if `.env` has key) |
| 4 | Clean, build features, save CSVs |
| 5 | Evaluate accuracy / precision / recall / F1 |
| 6 | Test recommendations |


In [ ]:
# --- Setup ---
import json, os, re, time, urllib.error, urllib.parse, urllib.request
from difflib import SequenceMatcher, get_close_matches
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, MultiLabelBinarizer, OneHotEncoder, StandardScaler

ROOT = Path(".").resolve()
LANGUAGE_FILES = ["telugu_movies.csv", "hindi_movies.csv", "english_movies.csv", "kannada_movies.csv"]
CACHE_DIR = ROOT / ".cache" / "tmdb"
CACHE_FILE = CACHE_DIR / "responses.json"
TMDB_BASE = "https://api.themoviedb.org/3"

def load_dotenv():
    env_path = ROOT / ".env"
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_dotenv()
HAS_TMDB_KEY = bool(os.environ.get("TMDB_API_KEY", "").strip())

# TMDB: auto-on when API key present. Set TMDB_LIMIT=100 to test a small batch.
RUN_TMDB_ENRICH = False  # set True only if language CSVs still need online fill
TMDB_LIMIT = None
WRITE_BACK_LANGUAGE_CSVS = True

print(f"Project root: {ROOT}")
print(f"TMDB enrichment: {'ON' if RUN_TMDB_ENRICH else 'OFF (add TMDB_API_KEY to .env)'}")



In [ ]:
# --- Constants ---
STANDARD_GENRES = [
    "action", "adventure", "animation", "comedy", "crime", "documentary", "drama",
    "family", "fantasy", "history", "horror", "music", "mystery", "romance",
    "science fiction", "thriller", "tv movie", "war", "western",
]

GENRE_ALIASES = {
    "ddrama": "drama", "drama thriller": "thriller", "thiller": "thriller",
    "sport": "action", "sports": "action", "sci-fi": "science fiction",
    "sci fi": "science fiction", "science-fiction": "science fiction",
    "sciencefiction": "science fiction", "musical": "music", "fiction": "drama",
    "emotional": "drama", "devotion": "drama", "devotional": "drama", "mythological": "history",
}

GENRE_KEYWORDS = {
    "action": ["action", "fight", "battle", "war", "combat", "explosion", "martial", "gun", "agent", "assassin", "revenge"],
    "adventure": ["adventure", "quest", "journey", "expedition", "treasure", "explorer"],
    "animation": ["animated", "animation", "cartoon"],
    "comedy": ["comedy", "comedic", "funny", "hilarious", "humor", "humour", "laugh"],
    "crime": ["crime", "criminal", "gangster", "heist", "police", "detective", "murder"],
    "documentary": ["documentary", "real-life", "biography of", "based on true"],
    "drama": ["drama", "emotional", "family conflict", "relationship", "struggle", "life of"],
    "family": ["family", "children", "kids", "parent"],
    "fantasy": ["fantasy", "magic", "mythical", "supernatural", "wizard", "dragon"],
    "history": ["historical", "history", "period", "revolution", "kingdom", "empire", "biopic"],
    "horror": ["horror", "ghost", "haunted", "demon", "possession", "zombie", "terrifying"],
    "music": ["music", "musical", "singer", "concert", "band", "song"],
    "mystery": ["mystery", "secret", "clue", "investigation", "whodunit", "missing"],
    "romance": ["romance", "romantic", "love story", "falls in love", "marriage", "heart"],
    "science fiction": ["sci-fi", "science fiction", "space", "future", "robot", "alien", "dystopian"],
    "thriller": ["thriller", "suspense", "conspiracy", "hostage", "chase", "kidnap"],
    "tv movie": ["tv movie", "television film"],
    "war": ["war", "soldier", "military", "battlefield", "army"],
    "western": ["western", "cowboy", "frontier"],
}

GENRE_DEFAULT_MOODS = {
    "comedy": ["joy", "amusement"], "romance": ["love", "desire"], "horror": ["fear", "surprise"],
    "thriller": ["fear", "confusion"], "action": ["excitement", "approval"], "drama": ["sadness", "neutral"],
    "family": ["approval", "caring"], "animation": ["joy", "amusement"], "adventure": ["excitement", "curiosity"],
    "crime": ["confusion", "anger"], "mystery": ["curiosity", "confusion"], "fantasy": ["excitement", "admiration"],
    "science fiction": ["curiosity", "excitement"], "music": ["joy", "admiration"],
    "documentary": ["realization", "neutral"], "history": ["admiration", "realization"],
    "war": ["sadness", "anger"], "western": ["neutral", "approval"],
}

MOOD_KEYWORDS = {
    "joy": ["happy", "joy", "celebration", "fun", "delight", "cheerful", "uplifting"],
    "amusement": ["funny", "hilarious", "comedy", "laugh", "humor"],
    "love": ["love", "romantic", "romance", "heart", "passion"],
    "desire": ["desire", "attraction", "longing", "crush"],
    "sadness": ["sad", "tragic", "death", "loss", "grief", "melancholy", "heartbreak"],
    "fear": ["fear", "terror", "horror", "scared", "nightmare", "danger"],
    "anger": ["angry", "rage", "revenge", "furious", "wrath"],
    "surprise": ["surprise", "shock", "twist", "unexpected", "stunned"],
    "confusion": ["confus", "mystery", "puzzle", "uncertain", "identity"],
    "excitement": ["thrilling", "exciting", "adrenaline", "intense", "epic"],
    "curiosity": ["discover", "investigation", "secret", "uncover", "explore"],
    "approval": ["inspiring", "hero", "triumph", "hope", "courage", "determined"],
    "caring": ["family", "protect", "care", "bond", "sacrifice"],
    "neutral": ["life", "story", "world", "man", "woman", "village", "city"],
    "realization": ["learns", "discovers", "realizes", "truth", "journey"],
    "optimism": ["hope", "dream", "future", "overcome"],
    "disappointment": ["fail", "betray", "disappoint", "broken"],
    "gratitude": ["thank", "grateful", "blessing"],
    "admiration": ["legend", "icon", "master", "brave", "honor"],
}
PRIMARY_MOODS = sorted(MOOD_KEYWORDS.keys())

TMDB_LANGUAGE_CODES = {"english": "en-US", "hindi": "hi-IN", "telugu": "te-IN", "kannada": "kn-IN"}
TMDB_GENRE_MAP = {
    28: "action", 12: "adventure", 16: "animation", 35: "comedy", 80: "crime", 99: "documentary",
    18: "drama", 10751: "family", 14: "fantasy", 36: "history", 27: "horror", 10402: "music",
    9648: "mystery", 10749: "romance", 878: "science fiction", 53: "thriller",
    10770: "tv movie", 10752: "war", 37: "western",
}


In [ ]:
# --- Helpers ---

def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return "" if text in {"", "unknown", "nan", "none", "na"} else text

def normalize_genre_token(token):
    cleaned = re.sub(r"\s+", " ", token.strip().lower())
    if not cleaned:
        return None
    cleaned = GENRE_ALIASES.get(cleaned, cleaned)
    return cleaned if cleaned in STANDARD_GENRES else None

def parse_genres(raw):
    if pd.isna(raw):
        return []
    genres = []
    for token in re.split(r"[,|/]", str(raw)):
        g = normalize_genre_token(token)
        if g and g not in genres:
            genres.append(g)
    return genres

def infer_genres_from_text(*texts):
    combined = " ".join(t for t in texts if t)
    if not combined:
        return []
    scores = {g: sum(1 for kw in kws if kw in combined) for g, kws in GENRE_KEYWORDS.items()}
    scores = {g: s for g, s in scores.items() if s}
    if not scores:
        return []
    top = max(scores.values())
    return [g for g, s in sorted(scores.items(), key=lambda x: (-x[1], x[0])) if s >= max(1, top - 1)]

def infer_moods_from_text(text):
    if not text:
        return []
    scores = {m: sum(1 for kw in kws if kw in text) for m, kws in MOOD_KEYWORDS.items()}
    scores = {m: s for m, s in scores.items() if s}
    if not scores:
        return []
    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:2]]

def infer_moods_from_genres(genres):
    moods = []
    for genre in genres:
        for mood in GENRE_DEFAULT_MOODS.get(genre, ["neutral"]):
            if mood not in moods:
                moods.append(mood)
            if len(moods) >= 2:
                return moods
    return moods or ["neutral"]

def combine_moods(overview, genres):
    moods = infer_moods_from_text(overview) or infer_moods_from_genres(genres) or ["neutral"]
    return ", ".join(moods[:2])

def primary_mood(moods):
    first = moods.split(",")[0].strip().lower()
    return first if first in PRIMARY_MOODS else "neutral"

def safe_median(series):
    med = series.median(skipna=True)
    return series.fillna(med) if pd.notna(med) else series


In [ ]:
# --- Load raw language CSVs ---

def load_raw_movies():
    frames = []
    for filename in LANGUAGE_FILES:
        path = ROOT / filename
        if not path.exists():
            continue
        df = pd.read_csv(path, encoding="utf-8-sig")
        df["Language"] = filename.split("_")[0]
        frames.append(df)
    if not frames:
        raise FileNotFoundError("No language CSV files found.")
    return pd.concat(frames, ignore_index=True)

raw = load_raw_movies()
print(f"Loaded {len(raw):,} rows")
missing = {
    "overview": raw["Overview"].isna().sum() + (raw["Overview"].astype(str).str.strip() == "").sum(),
    "genres": raw["Genres"].isna().sum() + (raw["Genres"].astype(str).str.strip() == "").sum(),
    "release_date": raw["Release Date"].isna().sum(),
}
print("Missing before enrich:", missing)
raw.head(3)


In [ ]:
# --- TMDB online enrichment ---

class TMDBClient:
    def __init__(self, delay=0.26):
        self.api_key = os.environ["TMDB_API_KEY"].strip()
        self.delay = delay
        self.cache = {}
        if CACHE_FILE.exists():
            try:
                self.cache = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
            except json.JSONDecodeError:
                self.cache = {}
        self._last = 0.0
        self._dirty = 0

    def _save_cache(self):
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        data = json.dumps(self.cache, ensure_ascii=False)
        try:
            tmp = CACHE_FILE.with_suffix(".tmp")
            tmp.write_text(data, encoding="utf-8")
            tmp.replace(CACHE_FILE)
        except (PermissionError, OSError):
            CACHE_FILE.write_text(data, encoding="utf-8")
        self._dirty = 0

    def _touch_cache(self):
        self._dirty += 1
        if self._dirty >= 25:
            self._save_cache()

    def _request(self, path, params, retries=5):
        for attempt in range(retries):
            elapsed = time.time() - self._last
            if elapsed < self.delay:
                time.sleep(self.delay - elapsed)
            url = f"{TMDB_BASE}{path}?" + urllib.parse.urlencode({**params, "api_key": self.api_key})
            req = urllib.request.Request(url, headers={"Accept": "application/json"})
            try:
                with urllib.request.urlopen(req, timeout=30) as resp:
                    self._last = time.time()
                    return json.loads(resp.read().decode("utf-8"))
            except urllib.error.HTTPError as exc:
                self._last = time.time()
                if exc.code in (404, 401):
                    return None
                if exc.code == 429 and attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                raise
            except (urllib.error.URLError, ConnectionResetError, TimeoutError, OSError):
                self._last = time.time()
                if attempt >= retries - 1:
                    return None
                time.sleep(min(30, 2 ** attempt))
        return None

    @staticmethod
    def pick_match(query_title, year, data):
        if not data or not data.get("results"):
            return None
        query = query_title.strip().lower()
        best, best_score = None, 0.0
        for item in data["results"][:12]:
            titles = [str(item.get(k, "")).lower() for k in ("title", "original_title") if item.get(k)]
            sim = max((SequenceMatcher(None, query, t).ratio() for t in titles), default=0.0)
            if year and not pd.isna(year) and item.get("release_date"):
                try:
                    diff = abs(int(str(item["release_date"])[:4]) - int(year))
                    sim += {0: 0.25, 1: 0.12}.get(diff, -0.2 if diff > 5 else 0)
                except ValueError:
                    pass
            if item.get("vote_count", 0) > 50:
                sim += 0.06
            elif item.get("vote_count", 0) > 5:
                sim += 0.02
            if sim > best_score:
                best_score, best = sim, item
        cutoff = 0.52 if len(query) <= 4 else 0.55
        return best if best_score >= cutoff else None

    def search(self, title, language, year=None):
        key = f"search|{language}|{year}|{title.strip().lower()}"
        if key in self.cache:
            return self.cache[key]
        params = {
            "query": title,
            "language": TMDB_LANGUAGE_CODES.get(language, "en-US"),
            "include_adult": "false",
            "region": "IN",
            "page": 1,
        }
        if year and not pd.isna(year):
            params["year"] = int(year)
        hit = self.pick_match(title, year, self._request("/search/movie", params))
        self.cache[key] = hit
        self._touch_cache()
        return hit

    def details(self, movie_id, language):
        key = f"detail|{movie_id}|{language}"
        if key in self.cache:
            return self.cache[key]
        data = self._request(f"/movie/{movie_id}", {"language": TMDB_LANGUAGE_CODES.get(language, "en-US")})
        self.cache[key] = data
        self._touch_cache()
        return data

def needs_tmdb(row):
    overview = str(row.get("Overview", "")).strip().lower()
    genres_ok = bool(parse_genres(row.get("Genres")))
    release = row.get("Release Date")
    missing_date = pd.isna(release) or str(release).strip() in {"", "nan", "NaT"}
    return overview in {"", "unknown", "nan", "none"} or not genres_ok or missing_date

def lookup_tmdb(client, title, lang, year):
    """Try multiple search strategies."""
    tries = [
        (lang, year),
        (lang, None),
        ("english", year),
        ("english", None),
    ]
    seen = set()
    for language, y in tries:
        key = (language, y)
        if key in seen:
            continue
        seen.add(key)
        hit = client.search(title, language, y)
        if hit:
            return hit, language
    return None, lang

def enrich_from_tmdb(df, limit=None):
    client = TMDBClient()
    out = df.copy()
    if "TMDB_Filled" not in out.columns:
        out["TMDB_Filled"] = False
    indices = [i for i, row in out.iterrows() if needs_tmdb(row)]
    print(f"Rows needing TMDB: {len(indices):,}")
    if limit:
        indices = indices[:limit]
    filled = not_found = 0
    for n, idx in enumerate(indices, 1):
        row = out.loc[idx]
        title = str(row["Title"]).strip()
        if not title:
            not_found += 1
            continue
        lang = str(row["Language"]).lower()
        dt = pd.to_datetime(row.get("Release Date"), errors="coerce")
        year = int(dt.year) if pd.notna(dt) else None
        hit, detail_lang = lookup_tmdb(client, title, lang, year)
        if not hit:
            not_found += 1
            continue
        detail = client.details(int(hit["id"]), detail_lang)
        if not detail:
            not_found += 1
            continue
        filled += 1
        ov = str(detail.get("overview", "")).strip()
        if ov and str(out.at[idx, "Overview"]).strip().lower() in {"", "unknown", "nan"}:
            out.at[idx, "Overview"] = ov
        if not parse_genres(out.at[idx, "Genres"]):
            genres = []
            for g in detail.get("genres", []):
                name = TMDB_GENRE_MAP.get(g.get("id")) or str(g.get("name", "")).lower()
                for pg in parse_genres(name):
                    if pg not in genres:
                        genres.append(pg)
            if genres:
                out.at[idx, "Genres"] = ", ".join(genres)
        rd = detail.get("release_date") or hit.get("release_date")
        if rd and (pd.isna(out.at[idx, "Release Date"]) or str(out.at[idx, "Release Date"]).strip() in {"", "nan"}):
            out.at[idx, "Release Date"] = rd
        if detail.get("vote_average") is not None:
            out.at[idx, "Rating"] = detail["vote_average"]
        if detail.get("vote_count") is not None:
            out.at[idx, "Vote_Count"] = detail["vote_count"]
        if detail.get("popularity") is not None:
            out.at[idx, "Popularity"] = detail["popularity"]
        out.at[idx, "TMDB_Filled"] = True
        if n % 50 == 0 or n == len(indices):
            print(f"TMDB {n}/{len(indices)} | filled={filled} | not_found={not_found}")
    client._save_cache()
    print(f"TMDB done: filled={filled}, not_found={not_found}")
    return out

if RUN_TMDB_ENRICH:
    print(f"Enriching via TMDB (limit={TMDB_LIMIT or 'all'})...")
    raw = enrich_from_tmdb(raw, limit=TMDB_LIMIT)
    if WRITE_BACK_LANGUAGE_CSVS:
        drop_cols = [c for c in ["TMDB_Filled"] if c in raw.columns]
        for filename in LANGUAGE_FILES:
            lang = filename.split("_")[0]
            part = raw[raw["Language"] == lang].drop(columns=drop_cols, errors="ignore")
            part.to_csv(ROOT / filename, index=False, encoding="utf-8-sig")
            print(f"Updated {filename} ({len(part):,} rows)")
else:
    print("Skipped TMDB (no API key in .env)")





In [ ]:
# --- Clean dataset ---

def clean_movies(df):
    out = df.copy()
    out["Title"] = out["Title"].map(clean_text)
    out["Language"] = out["Language"].map(clean_text)
    out["Cast"] = out["Cast"].map(clean_text)
    out["Overview"] = out["Overview"].map(clean_text)
    for col in ["Rating", "Popularity", "Vote_Count"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["Release Date"] = pd.to_datetime(out["Release Date"], errors="coerce", format="mixed")
    out["release_year"] = out["Release Date"].dt.year
    out = out[(out["Title"] != "") & (out["Language"] != "")]
    out = out.drop_duplicates(subset=["Title", "Language"], keep="first")

    out["Rating"] = out.groupby("Language")["Rating"].transform(lambda s: safe_median(s.replace(0, np.nan)))
    out["Vote_Count"] = out["Vote_Count"].fillna(0)
    out["Popularity"] = out.groupby("Language")["Popularity"].transform(safe_median)

    parsed = out["Genres"].apply(parse_genres)
    inferred = out.apply(
        lambda r: infer_genres_from_text(r["Title"], r["Overview"]) if not parsed.loc[r.name] else [],
        axis=1,
    )
    out["Genres_List"] = [p or i or ["drama"] for p, i in zip(parsed, inferred)]
    out["Genres"] = out["Genres_List"].apply(lambda gs: ", ".join(gs))
    out["Genres_Source"] = np.where(
        parsed.apply(len) > 0, "original",
        np.where(inferred.apply(len) > 0, "inferred_text", "fallback"),
    )
    out["Moods"] = out.apply(lambda r: combine_moods(r["Overview"], r["Genres_List"]), axis=1)
    out["Primary_Mood"] = out["Moods"].map(primary_mood)

    # Trim extreme outliers only
    out = out[out["Vote_Count"] <= out["Vote_Count"].quantile(0.995)]
    out = out[out["Popularity"] <= out["Popularity"].quantile(0.995)]
    return out.reset_index(drop=True)

cleaned = clean_movies(raw)
print(f"Cleaned rows: {len(cleaned):,}")
print(cleaned["Genres_Source"].value_counts())
print(f"Known release years: {cleaned['release_year'].notna().sum():,} / {len(cleaned):,}")
cleaned.head(3)



In [ ]:
# --- Build features & save ---

def build_features(df):
    result = df.copy()
    result["Popularity_Log"] = np.log1p(result["Popularity"].clip(lower=0))
    result["Vote_Count_Log"] = np.log1p(result["Vote_Count"].clip(lower=0))
    result["Popularity_Scaled"] = StandardScaler().fit_transform(result[["Popularity_Log"]])
    result["Vote_Count_Scaled"] = StandardScaler().fit_transform(result[["Vote_Count_Log"]])
    result["Rating_Scaled"] = StandardScaler().fit_transform(result[["Rating"]])
    # Fill missing years only for modelling using median of known years
    result["has_release_year"] = result["release_year"].notna().astype(int)
    result["release_year"] = result["release_year"].fillna(0)

    genres_df = pd.DataFrame(
        MultiLabelBinarizer(classes=STANDARD_GENRES).fit_transform(result["Genres_List"]),
        columns=STANDARD_GENRES,
    )
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    lang_df = pd.DataFrame(ohe.fit_transform(result[["Language"]]), columns=ohe.get_feature_names_out(["Language"]))
    mood_tokens = sorted({m.strip() for moods in result["Moods"] for m in moods.split(",") if m.strip()})
    mood_df = pd.DataFrame({
        f"mood_{m}": result["Moods"].apply(lambda t, mood=m: int(mood in str(t).lower()))
        for m in mood_tokens
    })
    base = result[[
        "Title", "Cast", "Overview", "Language", "Genres", "Primary_Mood", "Moods",
        "Genres_Source", "Rating", "release_year", "has_release_year", "Popularity_Scaled", "Vote_Count_Scaled", "Rating_Scaled",
    ]]
    return pd.concat([base, genres_df, mood_df, lang_df], axis=1)

features = build_features(cleaned)
cleaned.drop(columns=["Genres_List"]).to_csv(ROOT / "movies_cleaned.csv", index=False)
features.to_csv(ROOT / "movies_features.csv", index=False)

export = cleaned.drop(columns=["Genres_List", "release_year", "Genres_Source", "Primary_Mood", "Moods"]).copy()
export["Release Date"] = cleaned["Release Date"].dt.strftime("%Y-%m-%d")
export.loc[cleaned["Release Date"].isna(), "Release Date"] = ""
export.to_csv(ROOT / "movies.csv", index=False)
print("Saved movies_cleaned.csv, movies_features.csv, movies.csv")




In [ ]:
# --- Evaluate: accuracy, precision, recall, F1 ---

eligible = features[features["Primary_Mood"].isin(PRIMARY_MOODS)]
eligible = eligible[eligible["Overview"].astype(str).str.len() >= 20]
genre_cols = [g for g in STANDARD_GENRES if g in eligible.columns]
num_cols = ["Popularity_Scaled", "Vote_Count_Scaled", "Rating_Scaled", "release_year"]

tfidf = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2))
X = np.hstack([
    tfidf.fit_transform(eligible["Overview"].fillna("")).toarray(),
    eligible[genre_cols].fillna(0).to_numpy(),
    StandardScaler().fit_transform(eligible[num_cols].fillna(0)),
])
y = LabelEncoder().fit_transform(eligible["Primary_Mood"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

metrics_rows = []
best_pred = None
for k in [3, 5, 7, 9, 11, 15, 21, 31]:
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    row = {
        "k": k,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, pred, average="weighted", zero_division=0),
    }
    metrics_rows.append(row)
    if row["f1"] == max(r["f1"] for r in metrics_rows):
        best_pred, best_k = pred, k

metrics = pd.DataFrame(metrics_rows)
print(f"Samples: {len(X):,} | Best k={best_k}")
print(metrics.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nClassification report (best k):")
print(classification_report(y_test, best_pred, zero_division=0))
metrics


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for col in ["accuracy", "precision", "recall", "f1"]:
    ax.plot(metrics["k"], metrics[col], marker="o", label=col)
ax.set_xlabel("KNN k")
ax.set_ylabel("Score")
ax.set_title("Metrics vs k (should peak together near best k)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Recommender (memory-efficient: no full NxN matrix) ---

MOOD_ALIASES = {
    "happy": ["joy", "amusement", "excitement", "optimism", "approval"],
    "sad": ["sadness", "grief", "disappointment", "remorse"],
    "angry": ["anger", "annoyance", "disapproval"],
    "scared": ["fear", "nervousness", "surprise"],
    "romantic": ["love", "desire"],
    "curious": ["curiosity", "realization"],
    "confused": ["confusion", "realization"],
    "excited": ["excitement", "joy", "surprise"],
    "tired": ["neutral", "relief"],
    "neutral": ["neutral", "approval"],
}

class MovieRecommender:
    def __init__(self, path="movies_features.csv"):
        self.df = pd.read_csv(path)
        self.df["Moods"] = self.df["Moods"].fillna("neutral")
        for g in STANDARD_GENRES:
            if g not in self.df.columns:
                self.df[g] = 0
        mood_tokens = {t.strip().lower() for m in self.df["Moods"] for t in str(m).split(",") if t.strip()}
        self.mood_cols = [f"mood_{m}" for m in mood_tokens]
        for col in self.mood_cols:
            if col not in self.df.columns:
                self.df[col] = self.df["Moods"].apply(lambda t, mood=col[5:]: int(mood in str(t).lower()))
        self.df = self.df[self.df[STANDARD_GENRES].sum(axis=1) > 0].reset_index(drop=True)
        self.available_moods = sorted({c[5:] for c in self.mood_cols if self.df[c].sum() > 0})
        binary = self.df[STANDARD_GENRES + self.mood_cols].fillna(0).to_numpy(float) * 2.5
        numeric = self.df[["release_year", "Popularity_Scaled", "Vote_Count_Scaled", "Rating_Scaled"]].fillna(0).to_numpy(float)
        self.features = np.hstack([binary, MinMaxScaler().fit_transform(numeric)])

    def _scores(self, idx):
        return cosine_similarity(self.features[idx:idx+1], self.features).ravel()

    def resolve_mood(self, user_mood):
        q = user_mood.strip().lower()
        direct = get_close_matches(q, self.available_moods, n=1, cutoff=0.75)
        if direct:
            return direct
        alias = get_close_matches(q, MOOD_ALIASES.keys(), n=1, cutoff=0.75)
        if alias:
            labels = [m for m in MOOD_ALIASES[alias[0]] if f"mood_{m}" in self.df.columns and self.df[f"mood_{m}"].sum()]
            if labels:
                return labels
        return get_close_matches(q, self.available_moods, n=3, cutoff=0.6)

    def recommend_by_mood(self, user_mood, n=10, language=None):
        mapped = self.resolve_mood(user_mood)
        if not mapped:
            return mapped, pd.DataFrame()
        mask = np.zeros(len(self.df), bool)
        for m in mapped:
            col = f"mood_{m}"
            if col in self.df.columns:
                mask |= self.df[col].astype(bool).to_numpy()
        candidates = self.df[mask]
        if language:
            if "Language" in candidates.columns:
                candidates = candidates[candidates["Language"].str.lower() == language.lower()]
            else:
                col = f"Language_{language.lower()}"
                if col in candidates.columns:
                    candidates = candidates[candidates[col] == 1]
        if candidates.empty:
            return mapped, pd.DataFrame()
        score_col = f"mood_{mapped[0]}"
        seed_idx = candidates.sort_values([score_col, "Rating_Scaled"], ascending=False).index[0]
        pos = self.df.index.get_loc(seed_idx)
        ranked = candidates.index.to_series().map(lambda i: self.df.index.get_loc(i))
        ranked = ranked.sort_values(key=lambda s: s.map(lambda p: self._scores(pos)[p]), ascending=False).head(n)
        cols = [c for c in ["Title", "Moods", "Genres", "Rating", "Rating_Scaled", "release_year", "Language"] if c in self.df.columns]
        return mapped, self.df.loc[ranked.index][cols].reset_index(drop=True)

    def recommend_similar(self, title, n=10):
        titles = self.df["Title"].str.lower()
        q = title.strip().lower()
        match = self.df[titles == q]
        if match.empty:
            close = get_close_matches(q, titles.tolist(), n=1, cutoff=0.6)
            if not close:
                return pd.DataFrame()
            match = self.df[titles == close[0]]
        pos = self.df.index.get_loc(match.index[0])
        scores = self._scores(pos)
        order = np.argsort(scores)[::-1][1:n+1]
        cols = [c for c in ["Title", "Moods", "Genres", "Rating", "Rating_Scaled", "release_year", "Language"] if c in self.df.columns]
        return self.df.iloc[order][cols].reset_index(drop=True)

engine = MovieRecommender("movies_features.csv")
for mood in ["happy", "sad", "tired"]:
    mapped, recs = engine.recommend_by_mood(mood, n=5)
    print(f"\n{mood} -> {mapped}")
    if not recs.empty:
        print(recs.to_string(index=False))

